# **04 Parent Document Retriever와 청킹 전략**

### 학습 내용
1. Parent Document Retriever의 개념과 동작 원리
2. 청킹(Text Splitting) 전략
3. Child chunk로 검색 + Parent document 반환
4. RAG 성능 향상을 위한 검색 전략

**03_basic_rag_qdrant.ipynb 과의 차이:**
- 03.ipynb: 페이지 단위로 저장 및 검색 (청킹 없음)
- 04.ipynb: 페이지를 작은 chunk로 분할하여 검색, 전체 페이지 반환


**데이터**: 2026 주요업무계획.pdf

## 0. 환경 변수 설정

03에서 사용한 천안시 데이터를 활용하며, Qdrant Cloud를 사용합니다.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# .env는 노트북 파일이 아니라 Jupyter 커널의 현재 폴더를 기준으로 찾습니다.
env_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "smu-ai-service-bootcamp" / "rag-system" / ".env",
    Path.cwd() / "rag-system" / ".env",
]
env_path = next((path for path in env_candidates if path.is_file()), None)

if env_path is None:
    print("✗ .env 파일을 찾을 수 없습니다.")
else:
    load_dotenv(dotenv_path=env_path)
    print(f"환경 변수 파일: {env_path.resolve()}")

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

환경 변수 파일: C:\Users\taehy\Desktop\인공지능 기반 상명 AI Training\smu-ai-service-bootcamp\rag-system\.env
✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://14aa4220-1d44-461a-b29e-a4effdecccae.eu-west-2-0.aws.cloud.qdrant.io:6333


In [2]:
# .env 파일 로드
load_dotenv()

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY가 정상적으로 로드되었습니다.")
    print("API Key ?? ???? ????.")
else:
    print("OPENAI_API_KEY를 찾을 수 없습니다.")
    print(".env 파일을 생성하고 OPENAI_API_KEY를 설정해주세요.")

OPENAI_API_KEY가 정상적으로 로드되었습니다.
API Key: sk-proj-yw...K00A


## 1. Parent Document Retriever란?

### RAG의 딜레마

**문제 상황:**
- 작은 chunk: 검색은 정확하지만, 충분한 컨텍스트 부족
- 큰 chunk: 컨텍스트는 풍부하지만, 검색 정확도 떨어짐

### Parent Document Retriever의 해결책

**두 단계 접근:**
1. **검색 단계**: 작은 child chunk로 정확하게 검색
2. **반환 단계**: 해당 chunk가 속한 큰 parent 문서 반환

**장점:**
- 정확한 검색 + 풍부한 컨텍스트
- 문맥이 끊기지 않음
- LLM이 더 나은 답변 생성 가능

### 구조

```text
Parent Document (전체 페이지)
├── Child Chunk 1  ← vectorstore에 저장 (검색용)
├── Child Chunk 2  ← vectorstore에 저장
└── Child Chunk 3  ← vectorstore에 저장

검색 프로세스:
1. 사용자 질문 → Child Chunk 검색 (벡터 유사도)
2. 찾은 Child Chunk의 Parent ID 추출
3. Parent Document 반환 (전체 페이지)
```

## 2. Parent Document 로딩

03에서 사용한 천안시 주요업무계획 PDF를 사용합니다.
각 페이지가 Parent Document가 됩니다.

In [3]:
from langchain_core.documents import Document
import fitz

file_path = "../datasets/2026 주요업무계획.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (이것이 Parent)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "2026 주요업무계획.pdf",
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"  # Parent 식별자
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

총 423개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 1279자
평균 페이지 길이: 893자


## 3. Parent Document Retriever 구현

1. 페이지를 작은 Child chunk로 분할 (정확한 검색)
2. Child chunk를 vectorstore에 저장
3. Parent 문서를 docstore에 저장
4. Child로 검색, Parent 반환

### 3-1. Child Chunk 생성

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Child splitter: 검색용 작은 chunk 생성
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    # 각 parent를 chunk로 분할
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        # Child chunk에 parent_id 포함
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"], # parent 검색에 사용
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")


생성된 통계:
  - Parent 문서 수: 423
  - Child chunk 수: 1332
  - 평균 chunk/page: 3.1


In [5]:
# 모든 child chunk 내용 확인
for i, child_doc in enumerate(child_docs):
    print(f"Child Chunk {i + 1}:")
    print(f"  - Parent ID: {child_doc.metadata['parent_id']}")
    print(f"  - Page: {child_doc.metadata['page']}")
    print(f"  - Source: {child_doc.metadata['source']}")
    print(f"  - Content Length: {len(child_doc.page_content)}자")
    print(f"  - Content: {child_doc.page_content}")
    print("=" * 80)

Child Chunk 1:
  - Parent ID: page_3
  - Page: 3
  - Source: 2026 주요업무계획.pdf
  - Content Length: 126자
  - Content: 총괄현황01
         • 시정여건 및 운영방향   ·····················   5
         • 2026 주요업무 현황   ·····················   9

02  부서별 주요업무 현황
Child Chunk 2:
  - Parent ID: page_3
  - Page: 3
  - Source: 2026 주요업무계획.pdf
  - Content Length: 368자
  - Content: • 부시장직속   ·······································  15
         • 기획조정실   ·······································  31
         • 전략산업국   ·······································  59
         • 행정자치국   ·······································  89
         • 복지정책국   ······································· 115
         • 문화체육국   ······································· 153
Child Chunk 3:
  - Parent ID: page_3
  - Page: 3
  - Source: 2026 주요업무계획.pdf
  - Content Length: 356자
  - Content: • 농업환경국   ······································· 183
         • 건설안전교통국   ································ 217
         • 도시주택국   ····························

### 3-2. Vectorstore 구축 (Child Chunk 저장)

Child chunk를 Qdrant vectorstore에 저장합니다.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://14aa4220-1d44-461a-b29e-a4effdecccae.eu-west-2-0.aws.cloud.qdrant.io:6333


In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 새로운 컬렉션 생성 (child chunk 전용)
collection_name = "cheonan_child_chunks"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        # 컬렉션 재생성
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    # 컬렉션이 없으면 새로 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

컬렉션 'cheonan_child_chunks'이 이미 존재합니다.
컬렉션 'cheonan_child_chunks' 삭제 중...
컬렉션이 삭제되었습니다.
컬렉션 'cheonan_child_chunks' 생성 완료


In [8]:
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

vectorstore.add_documents(documents=child_docs)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")


1332개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


### 3-3. Docstore 구축 (Parent Document 저장)

Parent 문서를 별도로 저장합니다. 각 키(Key) 는 child chunks 의 부모를 찾을 때 사용됩니다.

```text
{'page_1' : Document(metadata={}, page_content=""),
'page_2' : Document(metadata={}, page_content=""),
'page_3' : Document(metadata={}, page_content=""), ... }
```

In [9]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 423개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_3', 'page_7', 'page_8', 'page_9', 'page_10']


In [15]:
parent_docstore

{'page_3': Document(metadata={'source': '2026 주요업무계획.pdf', 'page': 3, 'parent_id': 'page_3'}, page_content='    총괄현황01\n         • 시정여건 및 운영방향   ·····················   5\n         • 2026 주요업무 현황   ·····················   9\n\n02  부서별 주요업무 현황\n\n         • 부시장직속   ·······································  15\n         • 기획조정실   ·······································  31\n         • 전략산업국   ·······································  59\n         • 행정자치국   ·······································  89\n         • 복지정책국   ······································· 115\n         • 문화체육국   ······································· 153\n         • 농업환경국   ······································· 183\n         • 건설안전교통국   ································ 217\n         • 도시주택국   ······································· 251\n         • 보 건 소   ······································· 287\n         • 농업기술센터   ···································· 307\n         • 맑은물사업본부  ································· 327

### 3-4. Parent Document Retriever 클래스

직접 구현한 Parent Document Retriever입니다.

**동작 흐름:**
1. 사용자 질문 → Child chunk 검색 (vectorstore)
2. Child chunk의 parent_id 추출
3. parent_id로 Parent 문서 반환 (docstore)

In [10]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        """
        Args:
            vectorstore: Child chunk를 저장한 벡터스토어
            parent_docstore: Parent 문서를 저장한 dict
            k: 반환할 문서 수
        """
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        """
        질문으로 parent 문서 검색

        Args:
            query: 검색 질문

        Returns:
            Parent 문서 리스트
        """
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """
        비교용: Child chunk 직접 반환
        """
        return self.vectorstore.similarity_search(query, k=k)

In [11]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

## 4. 검색 동작 비교

천안시 관련 질문으로 Child chunk와 Parent document 검색을 비교합니다.

### 4-1. Child Chunk 직접 검색

In [12]:
query = "천안시의 환경 조성사업"

# Child chunk 검색
child_results = parent_retriever.get_child_chunks(query, k=2)

print(f"검색 쿼리: {query}")
print(f"\n[Child Chunk 검색] - 작은 chunk 반환")
print(f"결과: {len(child_results)}개\n")

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")
    print("-" * 80)

검색 쿼리: 천안시의 환경 조성사업

[Child Chunk 검색] - 작은 chunk 반환
결과: 2개


Chunk 1:
  페이지: 358
  Parent ID: page_358
  길이: 300자
  내용: 주요업무
❶ 핵심녹색 복합문화공간 「 천안정원」 조성
❷ 신규목재친화도시 목조전망대 조성사업
❸ 신규도솔문화공원 공원조성변경(      ) 계획 수립
❹ 신규녹색쌈지숲 조성사업
❺ 신규신방쉼터 정비공사
❻ 신규태조산길 가로수 정비사업
❼ 신규백석로 완충녹지 환경개선공사
❽ 신규풍세일반산업단지 도시계획시설경관녹지(            ) 정비공사
❾ 계속생활권역 실외정원 조성사업
❿ 계속기후대응도시숲 조성사업천안천(    일원)





                                               - 356 -
--------------------------------------------------------------------------------

Chunk 2:
  페이지: 205
  Parent ID: page_205
  길이: 203자
  내용: 정책       지속가능한 미래도시 천안을 위한목표
         신재생에너지 전환과 탄소중립



  세부전략
❍ 기후위기에 대응하는 탄소중립도시 조성
❍ 미세먼지 저감을 위한 그린 인프라 구축
❍ 대기질 개선으로 쾌적한 생활환경 조성
❍ 시민이 안전한 건강한 환경 복지 구현
❍ 산업단지 에너지자립으로 입주기업 에너지 효율 및 경쟁력 향상




  주요업무
--------------------------------------------------------------------------------


### 4-2. Parent Document 검색

Child chunk로 검색하고 전체 페이지를 반환합니다.

In [13]:
# Parent document 검색
parent_results = parent_retriever.invoke(query)

print(f"검색 쿼리: {query}")
print(f"\n[Parent Document 검색] - 전체 페이지 반환")
print(f"결과: {len(parent_results)}개\n")

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content}")
    print("-" * 80)

검색 쿼리: 천안시의 환경 조성사업

[Parent Document 검색] - 전체 페이지 반환
결과: 2개


Page 1:
  페이지 번호: 358
  Parent ID: page_358
  길이: 499자
  내용 미리보기: 정책   정원문화 진흥과 도시숲녹지·   확충으로 지속가능한목표
             녹색도시 조성



  세부전략

❍ 시민과 함께하는 정원문화 진흥지방정원,     조성을 통한 시민 휴식 공간 확충
❍ 쾌적하고 아름다운 정원, 도시숲 조성을 통한 생활권 녹색 공간 확충
❍ 깨끗하고 안전한 녹지환경을 위해 밀도 있는 환경정비 및 유지관리




  주요업무
❶ 핵심녹색 복합문화공간 「 천안정원」 조성
❷ 신규목재친화도시 목조전망대 조성사업
❸ 신규도솔문화공원 공원조성변경(      ) 계획 수립
❹ 신규녹색쌈지숲 조성사업
❺ 신규신방쉼터 정비공사
❻ 신규태조산길 가로수 정비사업
❼ 신규백석로 완충녹지 환경개선공사
❽ 신규풍세일반산업단지 도시계획시설경관녹지(            ) 정비공사
❾ 계속생활권역 실외정원 조성사업
❿ 계속기후대응도시숲 조성사업천안천(    일원)





                                               - 356 -
--------------------------------------------------------------------------------

Page 2:
  페이지 번호: 205
  Parent ID: page_205
  길이: 477자
  내용 미리보기: 정책       지속가능한 미래도시 천안을 위한목표
         신재생에너지 전환과 탄소중립



  세부전략
❍ 기후위기에 대응하는 탄소중립도시 조성
❍ 미세먼지 저감을 위한 그린 인프라 구축
❍ 대기질 개선으로 쾌적한 생활환경 조성
❍ 시민이 안전한 건강한 환경 복지 구현
❍ 산업단지 에너지자립으로 입주기업 에너지 효율 및 경쟁력 향상




  주요업무

❶ 핵심기후위기 대응을 위한 탄소

### 4-3. 비교 분석

**Child Chunk:**
- 정확한 키워드 매칭
- 작은 컨텍스트 (평균 400자)
- 문맥이 끊길 수 있음

**Parent Document:**
- 전체 페이지 반환 (평균 1000-3000자)
- 완전한 문맥 제공
- LLM이 더 정확한 답변 생성

In [14]:
print("[비교 분석]\n")

# Child chunk 통계
child_lengths = [len(doc.page_content) for doc in child_results]
print(f"Child Chunk:")
print(f"  - 문서 수: {len(child_results)}")
print(f"  - 평균 길이: {sum(child_lengths) / len(child_lengths):.0f}자")
print(f"  - 총 길이: {sum(child_lengths)}자")

# Parent document 통계
parent_lengths = [len(doc.page_content) for doc in parent_results]
print(f"\nParent Document:")
print(f"  - 문서 수: {len(parent_results)}")
print(f"  - 평균 길이: {sum(parent_lengths) / len(parent_lengths):.0f}자")
print(f"  - 총 길이: {sum(parent_lengths)}자")

print(f"\n컨텍스트 크기 비율: {sum(parent_lengths) / sum(child_lengths):.1f}배")

[비교 분석]

Child Chunk:
  - 문서 수: 2
  - 평균 길이: 252자
  - 총 길이: 503자

Parent Document:
  - 문서 수: 2
  - 평균 길이: 488자
  - 총 길이: 976자

컨텍스트 크기 비율: 1.9배


## 5. RAG 구현 (Parent Document Retriever 활용)

In [15]:
# ParentDocumentRetriever의 k 값 변경
parent_retriever.k = 2
print(parent_retriever.k)

2


In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# 프롬프트 템플릿 정의 (03 방식)
template = """
당신은 천안시 정책 전문가입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

<context>
{context}
</context>

<question>
{question}
</question>
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색 (child chunk로 검색, parent page 반환)
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\\n{doc.page_content}")

    context = "\\n\\n---\\n\\n".join(context_parts)

    print(f"\n[검색된 Parent 문서 수: {len(retrieved_docs)}]")
    print(f"\n[검색된 Parent 문서 내용]\n{context}")

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

In [17]:
# 테스트
question = "천안시의 환경 조성사업에 대해 상세히 설명해주세요"

print(f"질문: {question}\n")
answer = rag_with_parent_retriever(question)

display(Markdown(answer))

질문: 천안시의 환경 조성사업에 대해 상세히 설명해주세요


[검색된 Parent 문서 수: 2]

[검색된 Parent 문서 내용]
[출처: 2026 주요업무계획.pdf, 페이지: 358]\n정책   정원문화 진흥과 도시숲녹지·   확충으로 지속가능한목표
             녹색도시 조성



  세부전략

❍ 시민과 함께하는 정원문화 진흥지방정원,     조성을 통한 시민 휴식 공간 확충
❍ 쾌적하고 아름다운 정원, 도시숲 조성을 통한 생활권 녹색 공간 확충
❍ 깨끗하고 안전한 녹지환경을 위해 밀도 있는 환경정비 및 유지관리




  주요업무
❶ 핵심녹색 복합문화공간 「 천안정원」 조성
❷ 신규목재친화도시 목조전망대 조성사업
❸ 신규도솔문화공원 공원조성변경(      ) 계획 수립
❹ 신규녹색쌈지숲 조성사업
❺ 신규신방쉼터 정비공사
❻ 신규태조산길 가로수 정비사업
❼ 신규백석로 완충녹지 환경개선공사
❽ 신규풍세일반산업단지 도시계획시설경관녹지(            ) 정비공사
❾ 계속생활권역 실외정원 조성사업
❿ 계속기후대응도시숲 조성사업천안천(    일원)





                                               - 356 -\n\n---\n\n[출처: 2026 주요업무계획.pdf, 페이지: 205]\n정책       지속가능한 미래도시 천안을 위한목표
         신재생에너지 전환과 탄소중립



  세부전략
❍ 기후위기에 대응하는 탄소중립도시 조성
❍ 미세먼지 저감을 위한 그린 인프라 구축
❍ 대기질 개선으로 쾌적한 생활환경 조성
❍ 시민이 안전한 건강한 환경 복지 구현
❍ 산업단지 에너지자립으로 입주기업 에너지 효율 및 경쟁력 향상




  주요업무

❶ 핵심기후위기 대응을 위한 탄소중립 추진
❷ 신규전기차충전소 불법주차 스마트 단속기화재경보기·      설치
❸ 신규정온한 생활환경 조성을 위한 운행차 소음관리 강화
❹ 계속친환경 자동차 보급 확대
❺ 계속빛공해 없는 야간환경 조성
❻ 계속산업단지 에너지 

천안시의 **환경 조성사업**은 크게 보면 **탄소중립·대기환경 개선·녹색공간 확충·생활환경 정비**를 통해, 시민이 더 쾌적하고 안전하게 생활할 수 있는 도시를 만드는 사업입니다. 주어진 자료 기준으로 보면 아래와 같이 정리할 수 있습니다.

---

## 1) 신재생에너지 전환과 탄소중립을 위한 환경 조성
천안시는 **기후위기에 대응하는 탄소중립도시 조성**을 핵심 방향으로 두고 있습니다. 이를 위해 다음과 같은 사업을 추진합니다.

- **전기차충전소 불법주차 스마트 단속기화재경보기 설치**  
  전기차 충전구역의 불법주차를 줄이고, 화재 위험에 대비한 안전장치를 마련하는 사업입니다.
- **운행차 소음관리 강화**  
  도심의 정온한 생활환경을 위해 차량 소음 관리를 강화합니다.
- **친환경 자동차 보급 확대**  
  자동차 배출가스를 줄여 대기질 개선에 기여합니다.
- **빛공해 없는 야간환경 조성**  
  불필요한 조명을 줄여 야간 환경을 개선하고 생활 불편을 완화합니다.
- **산업단지 에너지 자급자족 인프라 구축 및 운영**  
  산업단지에 에너지 자립 기반을 마련해 기업의 에너지 효율과 경쟁력을 높입니다.
- **2026년 신재생에너지 융복합지원사업**  
  신재생에너지 활용을 확대해 에너지 전환을 지원합니다.
- **LPG용기 사용가구 시설개선사업**  
  취약한 에너지 사용환경을 개선하여 안전성과 효율성을 높입니다.

즉, 이 분야의 환경 조성사업은 단순히 “녹지 조성”이 아니라 **에너지, 대기, 소음, 조명, 안전**까지 포함하는 폭넓은 생활환경 개선사업입니다.

---

## 2) 정원문화 진흥과 도시숲 확충을 통한 녹색도시 조성
천안시는 시민 휴식과 도시 생태환경 개선을 위해 **정원문화와 도시숲·녹지 확충**을 추진하고 있습니다. 주요 내용은 다음과 같습니다.

- **핵심녹색 복합문화공간 「천안정원」 조성**  
  시민이 휴식하고 체험할 수 있는 대표적 녹색문화공간을 만드는 사업입니다.
- **목재친화도시 목조전망대 조성사업**  
  친환경 목재를 활용한 공간 조성으로 자연친화적 도시 이미지를 강화합니다.
- **도솔문화공원 공원조성변경 계획 수립**  
  공원의 기능과 공간 구성을 재정비하는 사업입니다.
- **녹색쌈지숲 조성사업**  
  생활권 내 작은 녹지를 늘려 도심 속 녹색쉼터를 확보합니다.
- **신방쉼터 정비공사**  
  기존 쉼터를 정비해 이용 편의와 환경을 개선합니다.
- **태조산길 가로수 정비사업**  
  가로수의 건강성과 경관을 개선해 안전하고 쾌적한 도로환경을 조성합니다.
- **백석로 완충녹지 환경개선공사**  
  도로 주변 완충녹지를 정비하여 소음·먼지 저감 효과를 높입니다.
- **풍세일반산업단지 도시계획시설 경관녹지 정비공사**  
  산업단지 주변 녹지를 개선해 경관과 환경을 동시에 향상시킵니다.
- **생활권역 실외정원 조성사업**  
  시민이 가까운 곳에서 자연을 접할 수 있도록 생활권 정원을 조성합니다.
- **기후대응도시숲 조성사업(천안천 일원)**  
  도시열섬 완화, 미세먼지 저감, 생태환경 개선을 위한 도시숲을 조성합니다.

이 사업들은 시민에게는 **휴식·치유 공간**을, 도시에는 **열섬 완화·미세먼지 저감·생태 다양성 확보** 효과를 줍니다.

---

## 3) 환경 조성사업의 공통 목표
자료에 따르면 천안시의 환경 조성사업은 다음 세 가지 방향으로 요약됩니다.

1. **시민과 함께하는 정원문화 진흥**  
   - 지방정원, 정원문화 공간을 통해 시민 휴식공간 확충

2. **쾌적하고 아름다운 도시숲·정원 조성**  
   - 생활권 녹색공간을 늘려 일상 속 녹색환경 제공

3. **밀도 있는 환경정비와 유지관리**  
   - 깨끗하고 안전한 녹지환경을 유지하여 지속가능한 도시 구현

---

## 4) 한눈에 보는 의미
천안시의 환경 조성사업은 다음과 같은 효과를 목표로 합니다.

- **대기질 개선**
- **탄소중립 실현**
- **미세먼지·소음·빛공해 저감**
- **도심 열섬 완화**
- **시민 휴식공간 확대**
- **산업단지와 주거지의 환경 개선**
- **안전하고 쾌적한 생활환경 조성**

즉, 천안시는 **에너지 전환형 환경정책**과 **녹색공간 조성정책**을 함께 추진하며, 도시 전반의 지속가능성을 높이려 하고 있습니다.

---

## 참고한 문서 출처
- **2026 주요업무계획.pdf, p. 205**  
  - 「신재생에너지 전환과 탄소중립」 정책 및 주요업무
- **2026 주요업무계획.pdf, p. 358**  
  - 「정원문화 진흥과 도시숲녹지 확충으로 지속가능한 녹색도시 조성」 정책 및 주요업무

원하시면 다음 단계로  
**1) 환경 조성사업을 분야별 표로 정리**하거나,  
**2) 천안시민 입장에서 체감효과 중심으로 다시 설명**해드릴 수 있습니다.

### 5-1. 다양한 질문 테스트

In [18]:
questions = [
    "천안시의 임산부 지원 정책은?",
    "천안시의 환경 조성사업을 요약해주세요",
    "민방위 체험교육 일정은 어떻게 되나요?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 천안시의 임산부 지원 정책은?


[검색된 Parent 문서 수: 2]

[검색된 Parent 문서 내용]
[출처: 2026 주요업무계획.pdf, 페이지: 131]\n Ⅳ 주요 업무

❶ 계속농촌형 양성평등 마을만들기 사업
  -  ( 기  간) 2026. 1. ~ 12.
  - (소요예산) 14 백만원 ( 시비)
  -  ( 대  상) 관내 읍· 면 마을 주민서북구(   · 동남구 각 2 개 마을)
  -  ( 내  용) 서로돌봄강화를 위한 마을별 양성평등교육 및 자기방어훈련 진행,
       천안시 제 5·6호 양성평등마을 MOU체결 후 현판식 진행 등

❷ 계속여성친화투어길 운영
  -  ( 기  간) 2026. 1. ~ 12.
  - (소요예산) 16 백만원 ( 시비)
  -  ( 대  상) 천안시민, 전국 여성친화도시 시민참여단 등
  -  ( 내  용) 천안시 여성 독립운동가 지속 발굴 ( 現 10 인보훈부,    등록),
       여성친화투어길 표지물 제작·설치, 시민·전국 시민참여단 대상
       여성 친화투어길 역사 해설 운영

❸ 계속임산부 이동편의 증진을 위한임산부“    교통비 지원”
  -  ( 기  간) 2026. 1. ~ 12.
  - (소요예산) 2,000백만원시비(      )
  -  ( 대  상) 천 안시 6개월 이상 거주 *임산부다문화가족(     포함)
                       * 임신 12주 이상 이거나 출산 후 3개월 이내인 자
  -  ( 내  용 ) 1인당 50만원 교통비 지급

  -  ( 지급방법 ) 임산부 교통비 전용 바우처카드지역화폐(             )
  -  ( 사 용 처) 자가용 유류비택시비,    사용가능

❹ 계속가족과 함께하는 일터천안시가족친화기관,‘               ’       재인증 추진

 -  ( 기  간 ) 2026. 4. ~ 11.
 - (내    용) 가족친화제도를 모범적으로 운영하는 공공기관 및 기업에 대하여
        심사를 통

천안시의 **임산부 지원 정책**은 다음과 같습니다.

1. **임산부 교통비 지원**
   - **지원대상**: 천안시에 **6개월 이상 거주한 임산부**(다문화가족 포함)  
     - 임신 **12주 이상**이거나 **출산 후 3개월 이내**인 자
   - **지원내용**: **1인당 50만 원** 교통비 지원
   - **지급방법**: 임산부 교통비 **전용 바우처카드(지역화폐)** 형태
   - **사용처**: **자가용 유류비, 택시비** 등
   - **사업기간**: 2026. 1. ~ 12.

2. **출생축하금 인상**
   - 임산부 직접 지원은 아니지만, 출산 가정의 경제적 부담을 줄이는 정책입니다.
   - **첫째 30만 원 → 100만 원**
   - **둘째 50만 원 → 100만 원**
   - **셋째아 이상 100만 원 → 1,000만 원(5년 분할 지급)**

3. **가족돌봄수당 지원**
   - 양육공백 가정에서 조부모 등 친족이 돌봄을 맡을 경우 지원하는 제도입니다.
   - **2~3세(24~47개월 이하) 영유아**를 둔 **기준중위소득 150% 이하 가정**
   - **월 30만 원** 지원
   - 임산부 본인보다는 출산 이후 양육 부담 완화에 도움이 되는 정책입니다.

### 참고한 문서
- **2026 주요업무계획.pdf, p.131**: 임산부 교통비 지원(대상, 금액, 지급방법, 사용처)
- **2026 주요업무계획.pdf, p.130**: 출생축하금 인상
- **2026 주요업무계획.pdf, p.128**: 가족돌봄수당 지원

원하시면 제가 이 내용을 **“임신·출산 단계별로 받을 수 있는 천안시 지원”** 형태로 정리해드릴게요.


질문: 천안시의 환경 조성사업을 요약해주세요


[검색된 Parent 문서 수: 2]

[검색된 Parent 문서 내용]
[출처: 2026 주요업무계획.pdf, 페이지: 358]\n정책   정원문화 진흥과 도시숲녹지·   확충으로 지속가능한목표
             녹색도시 조성



  세부전략

❍ 시민과 함께하는 정원문화 진흥지방정원,     조성을 통한 시민 휴식 공간 확충
❍ 쾌적하고 아름다운 정원, 도시숲 조성을 통한 생활권 녹색 공간 확충
❍ 깨끗하고 안전한 녹지환경을 위해 밀도 있는 환경정비 및 유지관리




  주요업무
❶ 핵심녹색 복합문화공간 「 천안정원」 조성
❷ 신규목재친화도시 목조전망대 조성사업
❸ 신규도솔문화공원 공원조성변경(      ) 계획 수립
❹ 신규녹색쌈지숲 조성사업
❺ 신규신방쉼터 정비공사
❻ 신규태조산길 가로수 정비사업
❼ 신규백석로 완충녹지 환경개선공사
❽ 신규풍세일반산업단지 도시계획시설경관녹지(            ) 정비공사
❾ 계속생활권역 실외정원 조성사업
❿ 계속기후대응도시숲 조성사업천안천(    일원)





                                               - 356 -\n\n---\n\n[출처: 2026 주요업무계획.pdf, 페이지: 340]\n정책
목표   미래를 위한 깨끗하고 안전한 하수처리 시스템 구축




  세부전략
❍ 도시 확장 및 환경기준 강화에 대비하여 처리 안정성 확보를 위한
  하수처리시설 단계적 증설 및 개량을 추진
❍ 하수관로 사업 확대을 통한 단계적 하수관로 보급률 향상 추진
❍ 지속 가능한 물 재이용 활성화




  주요업무

❶ 핵심 천안하수처리장 시설현대화 민간투자사업
❷ 핵심 천안시 통합바이오가스화시설 설치사업
❸ 계속 성환공공하수처리시설 기존시설 개량 및 4단계 증설 민간투자사업
❹ 계속 병천공공하수처리시설 1,2 단계 개량 및 3 단계 증설사업
❺ 계속성환 (3단계) 하수관로 정비공사
❻ 계속 천안처리구역 정비구역 해제지역 관로정비

천안시의 **환경 조성사업**은 크게 **녹색도시 조성**과 **하수처리·물재이용 기반 구축**으로 요약할 수 있습니다.

## 1) 정원문화 진흥과 도시숲·녹지 확충
천안시는 시민 휴식 공간과 생활권 녹색공간을 늘리기 위해 정원·도시숲·녹지 조성을 추진합니다.  
주요 사업은 다음과 같습니다.

- **핵심녹색 복합문화공간 「천안정원」 조성**
- **목재친화도시 목조전망대 조성사업**
- **도솔문화공원 공원조성변경 계획 수립**
- **녹색쌈지숲 조성사업**
- **신방쉼터 정비공사**
- **태조산길 가로수 정비사업**
- **백석로 완충녹지 환경개선공사**
- **풍세일반산업단지 도시계획시설 경관녹지 정비공사**
- **생활권역 실외정원 조성사업**
- **기후대응도시숲 조성사업(천안천 일원)**

이 사업들은 시민이 체감할 수 있는 **쾌적하고 아름다운 녹지공간 확대**, **휴식공간 확충**, **녹지환경 유지관리 강화**를 목표로 합니다.

## 2) 깨끗하고 안전한 하수처리 시스템 구축
도시의 환경기반을 개선하기 위해 하수처리시설을 확충·개량하고, 하수관로를 정비하며, 물 재이용도 확대합니다.  
주요 사업은 다음과 같습니다.

- **천안하수처리장 시설현대화 민간투자사업**
- **천안시 통합바이오가스화시설 설치사업**
- **성환공공하수처리시설 기존시설 개량 및 4단계 증설**
- **병천공공하수처리시설 1·2단계 개량 및 3단계 증설**
- **성환(3단계) 하수관로 정비공사**
- **천안처리구역 정비구역 해제지역 관로정비사업**
- **입장처리분구 하수관로 정비사업(1단계)**
- **성환처리구역 차집관로 정비공사**
- **성성호수공원 하수처리수 물재이용시설 설치사업**
- **성환공공하수처리시설 하수처리수 재이용시설 설치사업**

이 사업들은 **하수처리 안정성 확보**, **하수관로 보급률 향상**, **지속 가능한 물 재이용 활성화**를 목표로 합니다.

## 한마디로 정리하면
천안시는 **도시를 더 푸르고 쾌적하게 만드는 녹지·정원 조성사업**과 **안전하고 지속가능한 하수처리·재이용 기반 확충사업**을 함께 추진하고 있습니다.

### 참고 문서
- **2026 주요업무계획.pdf, p.358**: 정원문화 진흥과 도시숲·녹지 확충 관련 사업
- **2026 주요업무계획.pdf, p.340**: 하수처리 시스템 구축 및 물재이용 관련 사업


질문: 민방위 체험교육 일정은 어떻게 되나요?


[검색된 Parent 문서 수: 2]

[검색된 Parent 문서 내용]
[출처: 2026 주요업무계획.pdf, 페이지: 404]\nⅢ 가족과 함께하는 민방위 체험교육                 신규

 사 업 개 요

❍  ( 사업기간 ) 2026. 1. ~ 8.
❍  ( 사 업 비) 1 백만원 ( 시비)
❍  ( 사업대상 ) 4세 이상 자녀를 둔 민방위 대원(2~4년차가족)      10 ~ 15 가정
❍  ( 장  소) 충청남도 안전체험관
❍  ( 주요내용 ) 가족과 함께 사회재난자연재난응급처치실습,               ,      등 안전 체험 및 교육
            ※ 민방위 교육시간 인정

 추 진 계 획

❍안전체험관 사전 예약(’26. 1. ~ 2.) ➠ 세부 계획 수립(’26. 4.) ➠ 체험 교육 홍보,
 참가자 접수 (’26. 6.~ 7.) ➠민방위 안전체험 교육 실시(’26. 8.) / 여름방학기간 이용

 기 대 효 과

❍ 체험 위주의 실용적인 민방위 교육 강화
❍ 가족 안전의식 및 재난 대처 능력 고취


Ⅳ 모바일 행정전화 서비스 도입                    신규

 사 업 개 요

❍  ( 사업기간 ) 2026. 1. ~ 12.
❍  ( 사 업 비) 4.5백만원 ( 시비)
❍  ( 사 업 량) 동남구 직원 150명 내외출장외근이(       ,    잦은 직원 우선 )
❍  ( 주요내용 )
   - 공무 중 개인 휴대 전화 사용 시 상대방에 등록된 행정전화 번호 표시
   - 사무실 업무용PC로 문자송수신통화이력조회,                       , PC 통화 등 부가서비스 지원

 추 진 계 획

❍ 수요조사 및 대상자 선정(‘26. 1.) ➠ 서비스 운영(‘26. 2. ~ 6.) ➠ 평가 및 확대 검토(‘26. 7.)

 기 대 효 과

❍ 직원 개인정보 보호 및 업무의 효율성 및 유연성 증대

 협 조 사 항

❍ 동남구 각 과 및 읍면동

민방위 체험교육은 **2026년 8월에 실시**될 예정입니다.  
세부 추진계획에 따르면 **안전체험관 사전 예약(2026. 1. ~ 2.) → 세부 계획 수립(2026. 4.) → 참가자 접수 및 홍보(2026. 6. ~ 7.) → 민방위 안전체험 교육 실시(2026. 8.)** 순으로 진행됩니다. 또한 **여름방학 기간을 활용**할 계획입니다.

**참고 문서**
- `2026 주요업무계획.pdf`, **404쪽**

## 📖 과제 1: 다양한 질문으로 검색 및 답변(RAG) 테스트해보기

In [20]:
question = "천안시에서 계획하는 빵 관련 사업에는 어떤게 있나요?"

print(f"\n{'='*80}")
print(f"질문: {question}")
print(f"{'='*80}\n")

answer = rag_with_parent_retriever(question)
display(Markdown(answer))


질문: 천안시에서 계획하는 빵 관련 사업에는 어떤게 있나요?


[검색된 Parent 문서 수: 1]

[검색된 Parent 문서 내용]
[출처: 2026 주요업무계획.pdf, 페이지: 214]\nⅠ 복합문화축제로 도약하는 「2026 빵빵데이 천안」    핵심  계속

 사 업 개 요

❍  ( 사업기간 ) 2026. 1. ~ 12.
❍  ( 사 업 비) 450백만원 ( 도비100 시비 350)
❍  ( 주요내용 )
 - 제과업계 및 지역 농축산기업대학이,         ,    함께 만들어가는 빵산업 기반 구축
 - 천안의 우수한 빵 자원을 기반으로 문화관광과 연계한‘빵빵데이 천안’개최

 추 진 계 획

❍ 관광외식숙박,         ,   등과 연계된 ‘ 빵지순례 모바일 스탬프투어’ 개발
  - 사한국외식업중앙회천안시지부사대한숙박업중앙회천안시지회)                                                ,     )             등과 상생 협업
❍ ‘ 빵빵데이 천안 ’과 MICE 연계를 통한 축제 발전 방안 모색
 -‘ 빵의 도시 천안 ’의 역사와 정체성비전,   등 추구
  - 세미나 등을 통해 지역 산업 플랫폼으로 발전할 방안 강구
❍ 베이커리 경연대회 등 지역 산 · 학 ·연과 고품질 제과기술 발전방안 마련
❍ 세계 각국의 명품 빵을 소개하는 글로벌 베이커리 축제 확대 방안 모색
❍ 시민과 함께 공공적 가치 및 의미를 공유할 수 있도록 사회환원 사업 추구

 기 대 효 과

❍ 천안만의 차별화된 빵축제 개최로  ‵빵의도시 천안 ′ 의 브랜드 확립
❍지역산업 연계 강화를 통한 축제 지속가능성 확보 및 지역 경제 활성화 이바지

Ⅱ가족과 함께하는 “ 쌀맛나는 천안삼거리 떡 페스티벌 ”     핵심  계속

 사 업 개 요

❍  ( 사업기간 ) 2026. 7. ~ 12.
❍  ( 사 업 비) 30백만원 ( 도비15 시비 15)
❍  ( 주요내용 )
 - 전통먹거리떡(  )+지역농산물쌀(  )+체험・놀이가족(   

천안시가 2026년에 계획한 **빵 관련 사업**은 주로 **「2026 빵빵데이 천안」**입니다. 문서에 따르면 이 사업은 천안의 빵 자원을 기반으로 **문화관광과 연계한 복합문화축제**로 추진됩니다.

### 주요 내용
- **사업명:** 2026 빵빵데이 천안
- **사업기간:** 2026년 1월 ~ 12월
- **사업비:** 4억 5천만 원
- **핵심 추진 내용:**
  - 제과업계, 지역 농축산기업, 대학 등이 함께 만드는 **빵산업 기반 구축**
  - 천안의 우수한 빵 자원을 활용한 **문화관광 연계 축제 개최**
  - **빵지순례 모바일 스탬프투어** 개발
  - 관광·외식·숙박업계 등과 상생 협업
  - **MICE 연계**를 통한 축제 발전 방안 모색
  - **베이커리 경연대회** 등 산·학·연 협력
  - 세계 각국의 명품 빵을 소개하는 **글로벌 베이커리 축제 확대** 검토
  - 시민과 함께하는 **사회환원 사업** 추진

### 기대효과
- 천안만의 차별화된 빵축제를 통해 **‘빵의 도시 천안’ 브랜드 확립**
- 지역 산업 연계 강화로 **축제 지속가능성 확보** 및 **지역경제 활성화**

### 참고한 문서
- **출처:** 2026 주요업무계획.pdf
- **페이지:** 214페이지

원하시면 제가 이어서 **“빵빵데이 천안”의 세부 추진계획만 따로 정리**해드릴게요.

---

### 참고 자료

- [LangChain Text Splitters](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [Parent Document Retriever 개념](https://python.langchain.com/docs/modules/data_connection/retrievers/parent_document_retriever/)
- [Qdrant 공식 문서](https://qdrant.tech/documentation/)
- [Advanced RAG Techniques](https://python.langchain.com/docs/tutorials/rag/)